# KhazinaSmart — Data Loading & Cleaning

This notebook loads the Walmart Store Sales Forecasting dataset, performs initial inspection, merges dataframes, handles missing values, and saves a clean version for downstream analysis.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## 1. Load Raw Data

In [ ]:
# If Kaggle data not available, generate synthetic data
if not os.path.exists('../data/raw/train.csv'):
    print('Kaggle data not found — generating synthetic data...')
    import subprocess
    subprocess.run(['python', '../generate_sample_data.py'], check=True)

train = pd.read_csv('../data/raw/train.csv')
stores = pd.read_csv('../data/raw/stores.csv')
features = pd.read_csv('../data/raw/features.csv')
print(f'train:    {train.shape}')
print(f'stores:   {stores.shape}')
print(f'features: {features.shape}')

## 2. Initial Inspection

In [ ]:
for name, df in [('train', train), ('stores', stores), ('features', features)]:
    print(f'\n=== {name} ===')
    print(f'Shape: {df.shape}')
    print(f'Dtypes:\n{df.dtypes}')
    display(df.head(3))
    print(f'Nulls:\n{df.isnull().sum()}')

## 3. Merge DataFrames

In [ ]:
train['Date'] = pd.to_datetime(train['Date'])
features['Date'] = pd.to_datetime(features['Date'])

df = train.merge(stores, on='Store', how='left')
df = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
print(f'Merged shape: {df.shape}')
display(df.head(3))

## 4. Handle Missing Values

In [ ]:
md_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
print('Nulls BEFORE fill:')
print(df[md_cols].isnull().sum())
for col in md_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)
print('\nNulls AFTER fill:')
print(df[md_cols].isnull().sum())

## 5. Data Quality Report

In [ ]:
print('=== KhazinaSmart Data Quality Report ===')
print(f'Total rows:        {len(df):,}')
print(f'Date range:        {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Number of stores:  {df["Store"].nunique()}')
print(f'Number of depts:   {df["Dept"].nunique()}')
print(f'Holiday weeks (%): {df["IsHoliday"].mean()*100:.1f}%')
print(f'Weekly_Sales stats:')
print(f'  min:  {df["Weekly_Sales"].min():,.2f}')
print(f'  max:  {df["Weekly_Sales"].max():,.2f}')
print(f'  mean: {df["Weekly_Sales"].mean():,.2f}')

## 6. Save Clean Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/walmart_clean.csv', index=False)
print(f'Saved: {df.shape[0]:,} rows, {df.shape[1]} columns → data/processed/walmart_clean.csv')